In [ ]:
import torch
from collections import defaultdict
import matplotlib.pyplot as plt
from contextlib import contextmanager

class SD3Steering:
    """
    Steering implementation for Stable Diffusion 3 / 3.5 (MM-DiT models).
    Follows the same interface and logic as FluxSteering for compatibility with eval pipelines.
    """
    def __init__(self, pipe, device="cuda", n_steps=28):
        self.pipe = pipe
        self.device = device
        self.n_steps = n_steps

        # Locate Joint transformer blocks in SD3 / SD3.5
        # These are typically SD3TransformerBlock or JointTransformerBlock
        self.blocks = []
        for m in pipe.transformer.modules():
            name = m.__class__.__name__
            if name in ["SD3TransformerBlock", "JointTransformerBlock", "SD35TransformerBlock"]:
                self.blocks.append(m)

        # Fallback to direct attribute access if modular search is ambiguous
        if not self.blocks and hasattr(pipe.transformer, "transformer_blocks"):
             self.blocks = list(pipe.transformer.transformer_blocks)

        self.layer_idxs = list(range(len(self.blocks)))
        self.proj_layers = {}

        for li, block in enumerate(self.blocks):
            # Target the projection after joint attention.
            # In SD3/SD3.5 MM-DiT, the attention produces joint features that are projected.
            if hasattr(block, "attn") and hasattr(block.attn, "to_out"):
                # Handle both Sequential and ModuleList wrappers in diffusers
                if isinstance(block.attn.to_out, (torch.nn.ModuleList, torch.nn.Sequential)):
                    self.proj_layers[li] = block.attn.to_out[0]
                else:
                    self.proj_layers[li] = block.attn.to_out
            elif hasattr(block, "proj_out"):
                # Some architectures/versions use proj_out directly
                self.proj_layers[li] = block.proj_out

        print(f"Found {len(self.blocks)} transformer blocks")
        print(f"Hooking onto {len(self.proj_layers)} projection layers")

        # Helper state
        self._current_step = -1
        self._handles = []

    def _on_step_end(self, pipe, step, timestep, callback_kwargs):
        """Callback to track current step during generation."""
        self._current_step = int(step.item()) if torch.is_tensor(step) else int(step)
        return callback_kwargs

    def _clear_hooks(self):
        """Remove all active hooks."""
        for h in self._handles:
            h.remove()
        self._handles = []

    # =========================================================================
    # 1. VECTOR CREATION (LEARNING)
    # =========================================================================
    @torch.no_grad()
    def learn_vectors(self, pos_prompt, neg_prompt, seeds, top_k=15):
        """
        Learns and returns a dictionary of steering vectors using the difference-of-means method.
        """
        mean_diffs = defaultdict(lambda: defaultdict(float))
        counts = defaultdict(lambda: defaultdict(int))

        def collect_hook(layer_idx, sign):
            def hook(module, inputs, output):
                step = self._current_step + 1
                if 0 <= step < self.n_steps:
                    # In joint attention, output is usually a single tensor or a tuple
                    act = output[0] if isinstance(output, tuple) else output

                    # Mean over batch and sequence dimension
                    # output shape: (Batch, Seq, Dim) -> (Dim)
                    mean_act = act.detach().mean(dim=(0, 1))

                    mean_diffs[layer_idx][step] += (sign * mean_act)
                    counts[layer_idx][step] += 1
                return output
            return hook

        try:
            for seed in seeds:
                # Positive Pass (+1)
                self._clear_hooks()
                for li, proj in self.proj_layers.items():
                    # Prefix 'block_' used for unified layering in SD3
                    self._handles.append(proj.register_forward_hook(collect_hook(f"block_{li}", +1)))
                self._run_pipe_base(pos_prompt, seed)

                # Negative Pass (-1)
                self._clear_hooks()
                for li, proj in self.proj_layers.items():
                    self._handles.append(proj.register_forward_hook(collect_hook(f"block_{li}", -1)))
                self._run_pipe_base(neg_prompt, seed)

        finally:
            self._clear_hooks()

        # Process and Select Top-K candidates by gradient norm
        candidates = []
        for li in mean_diffs:
            for step in mean_diffs[li]:
                if counts[li][step] == 0: continue
                # (SumPos - SumNeg) / NumSeeds
                avg_diff = mean_diffs[li][step] / len(seeds)
                grad_norm = float(avg_diff.norm())
                candidates.append((grad_norm, li, step, avg_diff))

        candidates.sort(key=lambda x: x[0], reverse=True)
        top_candidates = candidates[:top_k]

        vectors = defaultdict(dict)
        print(f"--- Top {top_k} SD3 Steering Vectors ---")
        for rank, (strength, li, step, diff) in enumerate(top_candidates, start=1):
            direction = diff / (diff.norm() + 1e-8)
            vectors[li][step] = direction
            print(f"Rank {rank}: Layer {li}, Step {step}, Strength {strength:.4f}")

        return dict(vectors)

    # =========================================================================
    # 2. VECTOR APPLICATION (STEERING)
    # =========================================================================
    @contextmanager
    def apply_vectors(self, vectors, beta=10, clip=True):
        """
        Context Manager: Temporarily applies steering vectors to the model.
        """
        def steer_hook(layer_vectors):
            def hook(module, inputs, output):
                step = self._current_step + 1
                if step in layer_vectors:
                    is_tuple = isinstance(output, tuple)
                    target_output = output[0] if is_tuple else output

                    target_dir = layer_vectors[step].to(target_output.device, target_output.dtype)

                    # Score is the projection of current activation onto the steering vector
                    score = (target_output @ target_dir)
                    if clip:
                        score = torch.clamp(score, min=0.0)

                    # Update: activation - beta * score * direction
                    update = (beta * score).unsqueeze(-1) * target_dir.unsqueeze(0).unsqueeze(0)
                    new_output = target_output - update

                    if is_tuple:
                        return (new_output,) + output[1:]
                    return new_output

                return output
            return hook

        try:
            self._clear_hooks()
            for li, step_vecs in vectors.items():
                if isinstance(li, str) and li.startswith("block_"):
                    layer_idx = int(li.split("_")[1])
                    if layer_idx in self.proj_layers:
                        self._handles.append(self.proj_layers[layer_idx].register_forward_hook(steer_hook(step_vecs)))
            yield
        finally:
            self._clear_hooks()

    # =========================================================================
    # HELPERS
    # =========================================================================
    def _run_pipe_base(self, prompt, seed, steps=None):
        """Minimal pipe run helper for SD3/3.5"""
        steps = steps or self.n_steps
        self._current_step = -1
        g = torch.Generator(device=self.device).manual_seed(seed)

        return self.pipe(
            prompt=prompt,
            num_inference_steps=steps,
            generator=g,
            callback_on_step_end=self._on_step_end,
        ).images[0]

    def generate(self, prompt, seed, vectors=None, beta=10):
        """Convenience method to generate one image with optional steering"""
        if vectors:
            with self.apply_vectors(vectors, beta=beta):
                return self._run_pipe_base(prompt, seed)
        else:
            return self._run_pipe_base(prompt, seed)

    def sweep_betas(self, prompt, seed, vectors, betas=(0, 5, 10)):
        results = {}
        for b in betas:
            print(f"Generating BETA={b}...")
            img = self.generate(prompt, seed, vectors, beta=b)
            results[b] = img
        return results

if __name__ == "__main__":
    print("Class SD3Steering ready for Stable Diffusion 3.5.")
    # Example:
    # from diffusers import StableDiffusion3Pipeline
    # pipe = StableDiffusion3Pipeline.from_pretrained("stabilityai/stable-diffusion-3.5-large", ...)
    # steerer = SD3Steering(pipe)
    # vecs = steerer.learn_vectors("Van Gogh style", "neutral style", seeds=[1, 2])
    # img = steerer.generate("A dog in Van Gogh style", seed=42, vectors=vecs, beta=10)
